# Mugiwara4iOS — ONE PIECE: PIRATE WARRIORS 4

Pipeline isolado na branch `Mugiwara4iOS`, criado a partir de `SpkIOSzero`. Aceita base e update independentemente em `.nsp` ou `.xci`, baixa os dois links, usa `build_ipa_from_nsp.py` para extrair ExeFS/RomFS, gerar survey/perfil provisório, fazer rewrite ARM64 e enviar workspace/RomFS separados ao Drive com validação de tamanho e SHA-256.

Nenhuma Action é disparada automaticamente. O último passo é opt-in e envia o handoff exclusivamente para `Mugiwara4iOS`.


In [ ]:
# Configure os links quando os receber.
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
BASE_URL = ""       # link direto do pacote base
UPDATE_URL = ""     # link direto do update

# A extensão do nome precisa corresponder ao conteúdo real baixado.
# Base e update podem ter extensões diferentes: .nsp ou .xci.
BASE_FILENAME = "base.nsp"
UPDATE_FILENAME = "update.nsp"

# Deixe False para preparar e validar sem disparar GitHub Actions.
DISPATCH_TO_GITHUB = False

In [ ]:
from pathlib import Path
import argparse, json, os, subprocess, sys
from google.colab import auth

auth.authenticate_user()
ROOT = Path("/content/mugiwara4ios-repo")
if not ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "Mugiwara4iOS",
        "https://github.com/joao9vegeto-stack/KintTany.git", str(ROOT)
    ], check=True)
else:
    branch = subprocess.check_output(
        ["git", "-C", str(ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "Mugiwara4iOS":
        raise RuntimeError(f"Workspace existente está na branch {branch!r}; não vou trocar de branch.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "Mugiwara4iOS"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/Mugiwara4iOS"], check=True)

assert subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip() == "Mugiwara4iOS"
sys.path.insert(0, str(ROOT))
PY_FILES = [
    "colab/source_mugiwara.py", "colab/prepare_mugiwara.py", "colab/run_mugiwara.py",
    "scripts/prepare_mugiwara_actions.py", "scripts/build_mugiwara_shared.py",
]
subprocess.run([sys.executable, "-m", "py_compile", *[str(ROOT / item) for item in PY_FILES]], check=True)
print("Mugiwara4iOS Python syntax: OK")
print("Repository:", ROOT)
print("Branch:", "Mugiwara4iOS")

In [ ]:
from colab.run_mugiwara import prepare, recover

if BASE_URL.strip() and UPDATE_URL.strip():
    handoff = prepare(argparse.Namespace(
        drive_folder_id=DRIVE_FOLDER_ID,
        base_url=BASE_URL.strip(),
        update_url=UPDATE_URL.strip(),
        base_filename=BASE_FILENAME,
        update_filename=UPDATE_FILENAME,
    ))
else:
    print("Links ainda não preenchidos. Tentando recuperar uma preparação anterior do Drive...")
    handoff = recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID))
    if handoff is None:
        raise ValueError("Preencha BASE_URL e UPDATE_URL. Para XCI, use nomes terminados em .xci.")

print("\nPreparação recuperada/concluída:", handoff["request_id"])
print("Drive:", handoff["drive_output_folder_url"])

In [ ]:
# Relatório de validação do Colab antes de qualquer despacho para Actions.
prepared = handoff["prepared"]
print(json.dumps({
    "project": handoff["project"],
    "branch": handoff["branch"],
    "title_id": handoff["title_id"],
    "package_version": handoff["package_version"],
    "profile_type": prepared["profile_type"],
    "profile_source": prepared["profile_source"],
    "main_build_id": prepared["main_build_id"],
    "main_sha256": prepared["main_sha256"],
    "modules": prepared["modules"],
    "graphics_api": prepared["graphics_api"],
    "shader_source_flags": prepared["shader_source_flags"],
    "shader_source_reasons": prepared["shader_source_reasons"],
    "shader_source_audit": prepared["shader_source_audit"],
    "shader_chain": prepared["shader_chain"],
    "romfs_size": prepared["romfs_size"],
    "romfs_sha256": prepared["romfs_sha256"],
    "drive_workspace": handoff["work_archive"],
    "drive_romfs": handoff["game_data"]["romfs"],
}, indent=2))

In [ ]:
# DESPACHO EXPLÍCITO — só execute depois de revisar o handoff acima.
if DISPATCH_TO_GITHUB:
    from colab.run_mugiwara import dispatch
    dispatch(argparse.Namespace(handoff=Path("/content/mugiwara4ios/handoff.json")))
else:
    print("Actions NÃO disparada. Para enviar, altere DISPATCH_TO_GITHUB = True e execute esta célula.")